# Stage 1 — Learning-rate grid (validation split)

Tunes the base learning rate for each control arm: 10 epochs, seed 0, evaluated on the held-out
validation split (never the test set). `GRID` lists every lr tried per arm; finished runs are skipped,
so the table always shows each arm's whole grid.

Arms: `flat`, `flat_llrd` (slow-deep: layer4 slowest) and `flat_llrd_rev` (fast-deep: stem+layer1 slowest,
layer4 at the full lr). The reversed arm was added after the drift diagnostic showed that, under slow-deep,
layer4's drift is ~90% inherited from the faster layers below it.

**Selection rule (fixed before looking):** pick the lr with the highest `avg_incremental_acc`.
`final_avg_acc` is useless here (with Naive training every old task ends at 0%, so it is just last-task / 20),
and forgetting can't be the criterion because it rewards learning less.
If an arm's best lr is at the edge of its own grid, extend that arm's grid in that direction before choosing.

For both LLRD arms the tuned value is the **fast-level** lr; slower levels stay at lr/4 and lr/16.
Tuning each arm separately is option (c) from the "matched effective lr" question. If your supervisor
picks option (a) instead, LLRD just reuses Flat's best lr and you can skip the LLRD rows.

The Flat lr=0.03 point is the 10-epoch run from the epoch sweep (same config, so it is reused, not re-run).

In [ ]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # <-- EDIT ONCE
BRANCH = "main"

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if os.path.isdir(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, CODE], check=True)
print(subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
for _m in ["fyp_infra", "stage1_runner", "stage1_analysis", "stage1_drift_diagnostic"]:
    sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
# Hard stop for training: a CPU run is ~12x slower and its timings are useless for budgeting.
REQUIRE_GPU = True
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")

In [ ]:
EPOCHS = 10
ARMS = {
    "flat":          dict(lr_schedule="flat"),
    "flat_llrd":     dict(lr_schedule="llrd", level_ratios=(1, 4, 16), level_order="slow_deep"),
    "flat_llrd_rev": dict(lr_schedule="llrd", level_ratios=(1, 4, 16), level_order="fast_deep"),
}
# Every lr tried per arm (already-finished runs are skipped). Extend a list if its best is at an edge.
GRID = {
    "flat":          [0.01, 0.03, 0.1, 0.2, 0.3],
    "flat_llrd":     [0.01, 0.03, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9],
    "flat_llrd_rev": [0.03, 0.1, 0.2, 0.3, 0.5],   # new: 5 runs, ~45 GPU-min (estimate)
}

def grid_cfg(arm, lr):
    # The flat/0.03 point already exists as the epoch-sweep run; keeping its stage name
    # gives the identical config -> identical run_id -> run_sweep reuses that result.
    stage = "stage1_epoch_sweep" if (arm == "flat" and lr == 0.03) else "stage1_lr_grid"
    return sr.make_cfg(stage=stage, strategy="naive", epochs=EPOCHS, seed=0, lr=lr,
                       eval_split="val", **ARMS[arm])

CONFIGS = [grid_cfg(arm, lr) for arm in ARMS for lr in GRID[arm]]
SHOW = ["lr", "epochs", "eval_split"]
for c in CONFIGS: print(run_id(c), sr.arm_label(c), {k: c[k] for k in SHOW})

In [ ]:
out = run_sweep(CONFIGS, sr.run_experiment, paths)

In [ ]:
import numpy as np, pandas as pd
rows = []
for cfg in CONFIGS:
    r = load_result(cfg, paths)
    if r is None:
        rows.append({**{k: cfg[k] for k in SHOW}, "status": "not finished"}); continue
    m = r["metrics"]
    rows.append({"arm": sr.arm_label(cfg), **{k: cfg[k] for k in SHOW}, "run_id": r["run_id"],
                 "final_avg_acc": m["final_avg_acc"], "avg_forgetting": m["avg_forgetting"],
                 "avg_incremental_acc": m["avg_incremental_acc"],
                 # mean of the diagonal: how well each task was learned when it was current.
                 # Report next to forgetting: forgetting = peak - final, so it grows with this.
                 "acc_just_learned": float(np.mean(np.diag(m["acc_matrix"]))),
                 "steps": m["optimizer_steps"], "n_params": m["n_params"], "minutes": r["wall_clock_s"] / 60})
df = pd.DataFrame(rows); df

In [ ]:
LABEL = {sr.arm_label(sr.make_cfg(**a)): name for name, a in ARMS.items()}   # table label -> ARMS key
best = df.loc[df.groupby("arm")["avg_incremental_acc"].idxmax(), ["arm", "lr", "avg_incremental_acc", "acc_just_learned"]]
print(best)
for _, r in best.iterrows():
    lrs = GRID[LABEL[r["arm"]]]
    if r["lr"] in (min(lrs), max(lrs)):
        print(f"WARNING: best lr for {r['arm']} is at the edge of its grid ({r['lr']}); extend it before choosing.")